In [1]:
import sys
from pathlib import Path

# Get parent directory (Thesis-Edvin)
sys.path.append(str(Path.cwd().parent))

In [2]:
from utils import *
from datasets import load_dataset
from rag import *

In [15]:
# Load both splits
ds = load_dataset(
    "Eathus/github-issues-vul-max-detection-gpt-few-results-final3",
    split=["train", "test"]
)

# ds is now a list: [train_dataset, test_dataset]
train_ds, test_ds = ds

# Convert each to DataFrame
train_df = train_ds.to_pandas()
test_df = test_ds.to_pandas()

# Drop duplicates if needed (like you did for test)
train_df = train_df[~train_df.duplicated(subset="issue_github_id", keep=False)]
test_df = test_df[~test_df.duplicated(subset="issue_github_id", keep=False)]

# Combine both DataFrames
gpt_pred_df = pd.concat([train_df, test_df], ignore_index=True)

In [16]:
true_pos = gpt_pred_df[gpt_pred_df.gpt_is_relevant & ~gpt_pred_df.cve_id.isna()]
false_pos = gpt_pred_df[gpt_pred_df.gpt_is_relevant & gpt_pred_df.cve_id.isna()]
false_neg = gpt_pred_df[~gpt_pred_df.gpt_is_relevant & ~gpt_pred_df.cve_id.isna()]
all_true = gpt_pred_df[~gpt_pred_df.cve_id.isna()]

print("true pos:", len(true_pos))
print("false pos:", len(false_pos))
print("false neg:", len(false_neg))
print("all true:", len(all_true))
print("all", len(gpt_pred_df))

true pos: 3147
false pos: 1157
false neg: 263
all true: 3410
all 16030


In [5]:
cwes = load_dataset("Eathus/cwe_view1000_list_gpt_few_cwe_desc_final_best", split="train")
cwes_df = cwes.to_pandas()

# View CWE-1000

In [6]:
cwes_dict = cwes_df.to_dict('records')
documents_light = [
    create_ordered_cwe_document(weakness, old=True) for weakness in cwes_dict
]
document_dict = {doc.metadata["CWE_ID"]: doc for doc in documents_light}
documents = [
    Document(
        page_content=cwe.gpt_cwe_description,
        metadata=document_dict[cwe.ID].metadata,  # Original metadata
    )
    for _, cwe in cwes_df.iterrows()
]

allowed_usage = ["Allowed", "Allowed-with-Review", "Discouraged"]
documents = filter_cwe_docs_usage(documents, allowed_usage)

documents = split_docs(documents, echo=True, max_tokens=800)

Original: 881 docs
After splitting: 901 docs


In [7]:
vectorstore = create_vectorstore(documents, "tmp/faiss_enhanced_kb_index")

Loading existing vector store...


In [8]:
ranges = [1, 5, 10, 15, 20]
for k in ranges :
    rag_labeled_df = bare_rag_label(
        true_pos, vectorstore, match_col="gpt_vulnerability", k=k
    )
    evaluate_rag(rag_labeled_df, col='rag_label', save=True, md_filename=f'tmp/large_retrieval_results/classification_report_top-{k}.md')


100%|██████████| 3147/3147 [02:04<00:00, 25.30it/s]



✅ Markdown report saved to: tmp/large_retrieval_results/classification_report_top-1.md


100%|██████████| 3147/3147 [02:05<00:00, 25.18it/s]



✅ Markdown report saved to: tmp/large_retrieval_results/classification_report_top-5.md


100%|██████████| 3147/3147 [02:05<00:00, 25.04it/s]



✅ Markdown report saved to: tmp/large_retrieval_results/classification_report_top-10.md


100%|██████████| 3147/3147 [02:05<00:00, 25.09it/s]



✅ Markdown report saved to: tmp/large_retrieval_results/classification_report_top-15.md


100%|██████████| 3147/3147 [02:03<00:00, 25.41it/s]


✅ Markdown report saved to: tmp/large_retrieval_results/classification_report_top-20.md


# View CWE-1003

In [9]:
view1003 = load_dataset("Eathus/cwe_view1003_raw_list", split="train")
view1003_df = view1003.to_pandas()

In [10]:
documents1003 = [doc for doc in documents if doc.metadata['CWE_ID'] in view1003_df['ID'].tolist()]

In [11]:
vectorstore1003 = create_vectorstore(documents1003, "tmp/faiss_enhanced_kb1003_index")

Loading existing vector store...


In [12]:
ranges = [1, 5, 10, 15, 20]
for k in ranges :
    rag_labeled_df = bare_rag_label(
        true_pos, vectorstore1003, match_col="gpt_vulnerability", k=k
    )
    evaluate_rag(rag_labeled_df, col='rag_label', save=True, md_filename=f'tmp/large_retrieval_results/classification_report1003_top-{k}.md')

100%|██████████| 3147/3147 [02:03<00:00, 25.56it/s]



✅ Markdown report saved to: tmp/large_retrieval_results/classification_report1003_top-1.md


100%|██████████| 3147/3147 [02:03<00:00, 25.44it/s]



✅ Markdown report saved to: tmp/large_retrieval_results/classification_report1003_top-5.md


100%|██████████| 3147/3147 [02:02<00:00, 25.59it/s]



✅ Markdown report saved to: tmp/large_retrieval_results/classification_report1003_top-10.md


100%|██████████| 3147/3147 [02:02<00:00, 25.59it/s]



✅ Markdown report saved to: tmp/large_retrieval_results/classification_report1003_top-15.md


100%|██████████| 3147/3147 [02:03<00:00, 25.58it/s]


✅ Markdown report saved to: tmp/large_retrieval_results/classification_report1003_top-20.md
